In [1]:
print("HELLO")

HELLO


In [2]:
from langchain.tools import tool

@tool
def getWikiDocs(topic:str)->list:
    """Gets relevant docs from the wikipedia"""
    return []

@tool
def getYtTranscript(topic: str)->list:
    """Fetches relevant youtube video and gets the docs from the transcript """
    return []

@tool
def getUserDocs(topic: str)->list:
    """Fetches relevant docs from the documents provided by the user"""
    return []



In [3]:
tools = [getWikiDocs, getYtTranscript, getUserDocs]


In [4]:
from langchain_groq import ChatGroq
import os
key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b", api_key=key)

llm_with_tools = llm.bind_tools(tools=tools)

In [17]:
query = "Fetch me relevant docs from youtube on langgraph basics also fetch the notes i explicity provided"
response=llm_with_tools.invoke(query)

In [18]:
response.tool_calls

[{'name': 'getYtTranscript',
  'args': {'topic': 'langgraph basics'},
  'id': 'fc_bcab0c47-c7dc-42a5-a896-b2a90c0e4264',
  'type': 'tool_call'}]

In [2]:
from uuid import uuid8
uuid4()

UUID('f1db0a13-2091-46a7-a91a-816835c52ec7')

In [3]:
uuid8()

UUID('8d6f326e-7450-8d2b-b825-77dd874084e0')

In [6]:
import yt_dlp

ydl = yt_dlp.YoutubeDL({})

info = ydl.extract_info(
    "https://www.youtube.com/watch?v=exmSJpJvIPs",
    download=False
)

print("Language:", info.get("language"))
print("Subtitles:", info.get("subtitles"))
print("Auto captions:", info.get("automatic_captions"))

[youtube] Extracting URL: https://www.youtube.com/watch?v=exmSJpJvIPs
[youtube] exmSJpJvIPs: Downloading webpage


[youtube] exmSJpJvIPs: Downloading android vr player API JSON
Language: hi
Subtitles: {}
Auto captions: {'ab': [{'ext': 'json3', 'url': 'https://www.youtube.com/api/timedtext?v=exmSJpJvIPs&ei=pWY-apLeOs2M9fwPofGNwQU&caps=asr&opi=112496729&xoaf=5&xowf=1&hl=en&ip=0.0.0.0&ipbits=0&expire=1782499606&sparams=ip%2Cipbits%2Cexpire%2Cv%2Cei%2Ccaps%2Copi%2Cxoaf&signature=D41E345C1C997DB1744E604BB60EA3C917DF17EA.069752329CAA24AA9C76B0449CD981C098A67FDA&key=yt8&kind=asr&lang=hi&fmt=json3&tlang=ab', 'name': 'Abkhazian', 'impersonate': True, '__yt_dlp_client': 'android_vr'}, {'ext': 'srv1', 'url': 'https://www.youtube.com/api/timedtext?v=exmSJpJvIPs&ei=pWY-apLeOs2M9fwPofGNwQU&caps=asr&opi=112496729&xoaf=5&xowf=1&hl=en&ip=0.0.0.0&ipbits=0&expire=1782499606&sparams=ip%2Cipbits%2Cexpire%2Cv%2Cei%2Ccaps%2Copi%2Cxoaf&signature=D41E345C1C997DB1744E604BB60EA3C917DF17EA.069752329CAA24AA9C76B0449CD981C098A67FDA&key=yt8&kind=asr&lang=hi&fmt=srv1&tlang=ab', 'name': 'Abkhazian', 'impersonate': True, '__yt_dlp_

In [11]:
import yt_dlp
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter
from youtube_transcript_api import YouTubeTranscriptApi
import requests
import os 
from dotenv import load_dotenv

load_dotenv()


API_KEY = os.getenv("TRANSCRIPT_API_KEY")


class YTVideoFetcher:
    def __init__(self, topic, embedding_model, k=5):
        self.embedding_model = embedding_model
        
        self.topic = topic
        
        self.k = k
        self.imp_params = ['title', 'id', 'description', 'duration', 'view_count', 'like_count', 'webpage_url', 'language']
        self.url = f"ytsearch{k}:{self.topic}"
        self.ydl = yt_dlp.YoutubeDL({
            "queit":True
        })
        self.results = []
        self.search_videos()
        self.metadata = self.extract_metadata()

        

        

        

    def search_videos(self):

        self.results = self.ydl.extract_info(
            self.url,
            download=False
        )
    
    def extract_metadata(self):

        metadata = []

        for video in self.results["entries"]:

            video_data = {}

            for param in self.imp_params:
                video_data[param] = video.get(param)

            metadata.append(video_data)

        return metadata
    
    def filter_docs(self):
        topic_embedding = self.embedding_model.embed_query(
            self.topic
        )

        texts = [
            f"{doc['title']} {doc['description'][:500]}"
            for doc in self.metadata
        ]

        doc_embeddings = self.embedding_model.embed_documents(
            texts
        )

        for doc, emb in zip(self.metadata, doc_embeddings):
            doc["semantic_score"] = cosine_similarity(
                [topic_embedding],
                [emb]
            )[0][0]

        self.metadata.sort(
            key=lambda x: x["semantic_score"],
            reverse=True
        )

        self.metadata = self.metadata[: max(1, self.k // 2)]

        self.metadata.sort(
            key=lambda x: (
                x.get("view_count", 0),
                x.get("like_count", 0) or 0
            ),
            reverse=True
        )

        return self.metadata
    

    
    
    
    

    def get_transcripts(self)->dict:
        data = {}
        for video in self.metadata:

            video_id = video['id']
            
            url = 'https://transcriptapi.com/api/v2/youtube/transcript'
            params = {'video_url': video_id, 'format': 'json'}
            r = requests.get(url, params=params, headers={'Authorization': f'Bearer {API_KEY}'}, timeout=30)
            r.raise_for_status()
            transcript = r.json()['transcript']
            
            data[video_id] = transcript
        
        return data
        
        

    def chunk_transcript(self, data, max_chars=500):
        chunks = []

        for video_id, transcript in data.items():

            current_text = []
            start_time = None
            current_len = 0

            for seg in transcript:

                text = seg["text"]

                if start_time is None:
                    start_time = seg["start"]

                if current_len + len(text) > max_chars and current_text:

                    chunks.append({
                        "text": " ".join(current_text),
                        "start_time": start_time,
                        "end_time": seg["start"],
                        "video_id": video_id
                    })

                    current_text = []
                    start_time = seg["start"]
                    current_len = 0

                current_text.append(text)
                current_len += len(text)

            # Store the final chunk of this video
            if current_text:
                last_seg = transcript[-1]

                chunks.append({
                    "text": " ".join(current_text),
                    "start_time": start_time,
                    "end_time": last_seg["start"] + last_seg["duration"],
                    "video_id": video_id
                })

        return chunks
    
    def transcriber_chunker(self):
        data = self.get_transcripts()
        return self.chunk_transcript(data)


In [ ]:

YTF = YTVideoFetcher(topic="docker", embedding_model=Hugg)